In [ ]:
device = "0"
model_name = "meta-llama/Llama-3.2-3B"
dataset_size = 200
evaluate_scores = True

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [ ]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "varying_length_long"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

Using device: cuda:1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/yangkun/anaconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Model: meta-llama/Llama-3.2-3B
Loading dataset from cache...


{'input_ids': tensor([[128000,    791,  59796,   6406,   8925,   6787,    311,   7055,   7884,
          13658,    389,   7231,   8339,    400,   1419,   3610,    323,   1023,
          36580,    311,   1977,    264,  26097,  15679,    304,  29016,   4409,
             11,    719,   1193,   1306,  11011]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
from utils import (
    load_default_wepa_watermarker,
    run_experiment,
    get_wat_name,
    load_default_exp_watermarker,
)

wats = [
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=1,
    ),
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=2,
    ),
    load_default_exp_watermarker(
        model.config.vocab_size,
        device,
    ),
]
max_lengths = [25, 50, 75, 100, 125, 150, 175, 200, 225, 250]

for wat in wats:
    for max_length in max_lengths:
        wat_name = get_wat_name(wat)

        run_experiment(
            experiment=experiment,
            run_name=f"{model.name_or_path}_{wat_name}",
            wat=wat,
            model=model,
            tokenizer=tokenizer,
            data=data,
            max_length=max_length,
            device=device,
            evaluate_scores=evaluate_scores,
            upperbound=True,
        )